# 02. Population and fertilizer

**Purpose:** production tripled, but the population more than doubled. I need population
to tell those apart. Fertilizer is for the comparison in notebook 03.

**Input:** `crop_panel.csv` and the World Bank API

**Outputs:** `crop_panel_with_pop.csv`, `nigeria_per_capita.csv`, `fertilizer.csv`

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
import numpy as np

from src import config as C
from src import io_utils as io

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

crops = pd.read_csv(C.PROCESSED / "crop_panel.csv")
print("crop panel:", crops.shape)

crop panel: (6591, 6)


## 1. Get population

The World Bank API is free and needs no account. `fetch_wdi` stops if the answer comes
back split over pages, because then I would only have part of the data.

In [2]:
pop = io.fetch_wdi("SP.POP.TOTL").rename(columns={"value": "population"})
print("rows:", len(pop))
pop.head(3)

rows: 16675


,iso3,wb_country,year,population
0,AFE,Africa Eastern and Southern,2024,769280888.0
1,AFE,Africa Eastern and Southern,2023,750491370.0
2,AFE,Africa Eastern and Southern,2022,731821393.0


## 2. Drop the aggregates

The API returns World, Euro area, Sub-Saharan Africa and about seventy-five others mixed
in with real countries.

They are a problem because they contain the countries. Nigeria is inside Sub-Saharan
Africa, inside Low and middle income, inside World.

Filtering on the length of the country code does not work, because `WLD` is three
letters. The World Bank labels them itself, so I use that.

In [3]:
real = io.real_country_codes()
before = len(pop)
pop = pop[pop["iso3"].isin(real)]

print("real countries:", len(real))
print(f"rows {before:,} -> {len(pop):,}")

real countries: 217
rows 16,675 -> 13,859


## 3. Fix the names

FAOSTAT and the World Bank spell three of my fifteen differently. `Côte d'Ivoire` and
`Cote d'Ivoire` look the same but do not match.

In [4]:
print("not found:", [p for p in C.PEERS if p not in set(pop["wb_country"])])
print()
for old, new in C.WB_NAME_FIX.items():
    print(f"  {old:<20} -> {new}")

not found: ["Côte d'Ivoire", 'United Republic of Tanzania', 'Egypt']

  Egypt, Arab Rep.     -> Egypt
  Tanzania             -> United Republic of Tanzania
  Cote d'Ivoire        -> Côte d'Ivoire


In [5]:
pop["country"] = pop["wb_country"].replace(C.WB_NAME_FIX)

still_missing = [p for p in C.PEERS if p not in set(pop["country"])]
if still_missing:
    raise ValueError(f"still not matching: {still_missing}")

print("all 15 names match now")

all 15 names match now


I change the World Bank names to match FAOSTAT, not the other way round, because the
crop panel is already built on FAOSTAT names.

`.replace()` and not `.map()`, because map would blank every country I did not list.

## 4. Join

A join is where this kind of work goes wrong quietly. If the right side has a duplicate,
rows on the left get copied and every total built from them is too big, with no error.

Three guards: keep all crop rows, tell pandas to expect many-to-one, and check the row
count did not change.

In [6]:
before = len(crops)

merged = crops.merge(pop[["country", "year", "population"]],
                     on=["country", "year"], how="left",
                     validate="many_to_one", indicator=True)

print(f"rows {before:,} -> {len(merged):,}")
if len(merged) != before:
    raise ValueError("the join duplicated rows")

print()
print(merged["_merge"].value_counts().to_string())

rows 6,591 -> 6,591

_merge
both          6591
left_only        0
right_only       0


In [7]:
unmatched = merged.loc[merged["_merge"] == "left_only", ["country", "year"]].drop_duplicates()
print("crop rows with no population:", len(unmatched))

merged = merged.drop(columns="_merge")

crop rows with no population: 0


The same population repeats across a country's eight crop rows. That is fine. Reusing
a value is what I want, duplicating rows is the problem, and the row count tells me which
happened.

## 5. Food per person

In [8]:
ng_prod = merged[merged["country"] == "Nigeria"].groupby("year")["production_t"].sum().reset_index()
ng_pop  = pop.loc[pop["country"] == "Nigeria", ["year", "population"]]

per_capita = ng_prod.merge(ng_pop, on="year", how="inner", validate="one_to_one")
per_capita["kg_per_person"] = per_capita["production_t"] * 1000 / per_capita["population"]
per_capita = per_capita[per_capita["year"].between(C.START_YEAR, 2024)].reset_index(drop=True)

per_capita.head(3).round(1)

,year,production_t,population,kg_per_person
0,1990,52774008.0,97120925.0,543.4
1,1991,64187000.0,99720162.0,643.7
2,1992,71183000.0,102372770.0,695.3


In [9]:
s = per_capita.set_index("year")["kg_per_person"]

print(f"{s.index[0]}  {s.iloc[0]:.0f} kg")
print(f"{s.idxmax()}  {s.max():.0f} kg   <- highest")
print(f"{s.index[-1]}  {s.iloc[-1]:.0f} kg")
print()
print(f"up to the peak:   {(s.max() / s.iloc[0] - 1) * 100:+.1f}%")
print(f"peak to now:      {(s.iloc[-1] / s.max() - 1) * 100:+.1f}%")

1990  543 kg
2006  792 kg   <- highest
2024  709 kg

up to the peak:   +45.7%
peak to now:      -10.5%


Food per person rose for sixteen years and then stopped.

The line is noisy though. 2018 got within 3% of the 2006 peak, so I would not say it
collapsed. It stopped improving.

In [10]:
recent = s[s.index >= 2010]
print(f"since 2010: low {recent.min():.0f}, high {recent.max():.0f}")
print(f"2018: {s.loc[2018]:.0f} kg, {(s.loc[2018] / s.max() - 1) * 100:.1f}% off the peak")

since 2010: low 609, high 768
2018: 768 kg, -3.0% off the peak


## 6. Fertilizer

Same window as the yield comparison in notebook 03, so the two line up.

In [11]:
fert = io.fetch_wdi("AG.CON.FERT.ZS", *C.REF_WINDOW).rename(columns={"value": "fert_kg_ha"})
fert = fert[fert["iso3"].isin(real)]
fert["country"] = fert["wb_country"].replace(C.WB_NAME_FIX)

fertilizer = (fert[fert["country"].isin(C.PEERS)]
              .groupby("country")["fert_kg_ha"].mean()
              .reset_index()
              .sort_values("fert_kg_ha", ascending=False)
              .reset_index(drop=True))
fertilizer.round(1)

,country,fert_kg_ha
0,Egypt,576.6
1,South Africa,102.2
2,Kenya,49.7
3,Ethiopia,38.2
4,Ghana,37.3
5,Côte d'Ivoire,36.7
6,Benin,35.1
7,Mali,17.7
8,Nigeria,15.8
9,Senegal,15.6


In [12]:
ng = fertilizer.loc[fertilizer["country"] == "Nigeria", "fert_kg_ha"].values[0]
print(f"Nigeria: {ng:.1f} kg/ha\n")
for c in ["Egypt", "South Africa", "Kenya", "Ethiopia", "Ghana"]:
    v = fertilizer.loc[fertilizer["country"] == c, "fert_kg_ha"].values[0]
    print(f"  {c:<14} {v:6.1f}   {v / ng:5.1f}x Nigeria")

Nigeria: 15.8 kg/ha

  Egypt           576.6    36.5x Nigeria
  South Africa    102.2     6.5x Nigeria
  Kenya            49.7     3.2x Nigeria
  Ethiopia         38.2     2.4x Nigeria
  Ghana            37.3     2.4x Nigeria


Nigeria is near the bottom. That fits with low yields but does not prove anything on
its own, so notebook 04 tests it properly.

## 7. Save

In [13]:
merged.to_csv(C.PROCESSED / "crop_panel_with_pop.csv", index=False)
per_capita.to_csv(C.PROCESSED / "nigeria_per_capita.csv", index=False)
fertilizer.to_csv(C.PROCESSED / "fertilizer.csv", index=False)

for f in ["crop_panel_with_pop.csv", "nigeria_per_capita.csv", "fertilizer.csv"]:
    print(f"{f:<28}", pd.read_csv(C.PROCESSED / f).shape)

crop_panel_with_pop.csv      (6591, 7)
nigeria_per_capita.csv       (35, 4)
fertilizer.csv               (15, 2)


---

**Next:** `03_analysis.ipynb`.